# TASK 4: Speech-to-Reasoning Pipeline
# Platform: Google Colab (T4 GPU)

In [5]:
# ──────────────────────────────────────────
# CELL 1: Install Libraries
# ──────────────────────────────────────────
!pip install openai-whisper
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install ffmpeg-python soundfile gradio
!pip install --upgrade gradio typer click
!apt-get install -y ffmpeg   # ← Needed for audio processing

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-rbfdwi3f/unsloth_96be890b354545e4bfa734a473553cbe
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-rbfdwi3f/unsloth_96be890b354545e4bfa734a473553cbe
  Resolved https://github.com/unslothai/unsloth.git to commit 7ef8cde3c22a3bc2240353353caa21051a42ea90
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstall

Reading package lists... Done
^C


In [1]:
# ──────────────────────────────────────────
# CELL 2: Import Libraries
# ──────────────────────────────────────────

import torch
import whisper
import numpy as np
from unsloth import FastLanguageModel
import gradio as gr
import warnings
warnings.filterwarnings("ignore")

print(" All libraries imported!")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
 All libraries imported!
PyTorch version: 2.10.0+cu128
GPU available: True


In [2]:
# ──────────────────────────────────────────
# CELL 3: Load Whisper Speech-to-Text Model
# ──────────────────────────────────────────

"""
Whisper is OpenAI's speech recognition model.
Models: tiny, base, small, medium, large
Bigger = more accurate but slower.
'base' is a good balance for Colab.
"""

print("Loading Whisper model...")

# Load the 'base' Whisper model
# It can transcribe audio in 99 languages!
whisper_model = whisper.load_model("base")

print(" Whisper loaded!")
print(f"Whisper model type: base")
print(f"Whisper running on: {'GPU' if next(whisper_model.parameters()).is_cuda else 'CPU'}")

Loading Whisper model...
 Whisper loaded!
Whisper model type: base
Whisper running on: GPU


In [3]:
# ──────────────────────────────────────────
# CELL 4: Load Quantized LLM
# ──────────────────────────────────────────

print("\nLoading LLM (this takes 2-3 minutes)...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,   # 4-bit quantization for less memory
)

# Set to inference mode
FastLanguageModel.for_inference(model)

print(" LLM loaded!")



Loading LLM (this takes 2-3 minutes)...
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


 LLM loaded!


In [4]:
# ──────────────────────────────────────────
# CELL 5: Create Sample Audio File (for testing)
# ──────────────────────────────────────────

"""
Since you may not have an audio file, we create a test audio file
using text-to-speech. In real use, you'd record your own audio.
"""


!pip install gTTS

from gtts import gTTS
import os

# Create sample audio with a medical question
sample_text = "What are the main symptoms of type 2 diabetes and how is it treated?"

# Convert text to audio file
tts = gTTS(text=sample_text, lang='en', slow=False)
tts.save("/content/sample_question.mp3")

print(f" Sample audio created: /content/sample_question.mp3")
print(f"Audio content: '{sample_text}'")

  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
Using cached click-8.1.8-py3-none-any.whl (98 kB)
  Attempting uninstall: click
    Found existing installation: click 8.3.2
    Uninstalling click-8.3.2:
      Successfully uninstalled click-8.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


 Sample audio created: /content/sample_question.mp3
Audio content: 'What are the main symptoms of type 2 diabetes and how is it treated?'


In [5]:
import whisper
import torch

if 'whisper_model' not in locals():
    print("Loading Whisper model in Cell 6...")
    whisper_model = whisper.load_model("base")
    print("Whisper model loaded.")

def transcribe_audio(audio_path):
    """
    Converts audio file to text using Whisper.

    Parameters:
        audio_path: path to the audio file (.mp3, .wav, .m4a, etc.)

    Returns:
        Transcribed text as string, or an error message if transcription fails.
    """

    print(f" Transcribing: {audio_path}")

    try:
        # Whisper transcribes the audio
        result = whisper_model.transcribe(
            audio_path,
            language="en",          # Specify English (or None for auto-detect)
            task="transcribe",       # 'transcribe' = keep original language
            fp16=torch.cuda.is_available()  # Use fp16 on GPU for speed
        )
        transcribed_text = result["text"].strip()
        print(f" Transcription done!")
        print(f"Transcribed: '{transcribed_text}'")
        return transcribed_text
    except Exception as e:
        error_message = f"ERROR during transcription: {e}"
        print(error_message)
        return error_message

# Test transcription
transcribed = transcribe_audio("/content/sample_question.mp3")

 Transcribing: /content/sample_question.mp3
 Transcription done!
Transcribed: 'What are the main symptoms of type 2 diabetes and how is it treated?'


In [6]:
# ──────────────────────────────────────────
# CELL 7: Generate Response with LLM
# ──────────────────────────────────────────

def generate_llm_response(question, max_tokens=300):
    """
    Sends transcribed text to LLM and gets a reasoned response.

    Parameters:
        question: the transcribed text from Whisper
        max_tokens: maximum length of the response

    Returns:
        LLM's response as string
    """

    # Create a clear prompt with the transcribed question
    prompt = f"""You are a knowledgeable and helpful assistant.
The following question was spoken aloud and transcribed.
Please answer it clearly and thoroughly.

Question (from speech): {question}

Detailed Answer:"""

    # Tokenize
    inputs = tokenizer(
        [prompt],
        return_tensors="pt"
    ).to("cuda")

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            use_cache=True,
            repetition_penalty=1.1,   # Avoid repeating the same sentences
        )

    # Decode response
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the answer part
    if "Detailed Answer:" in full_text:
        response = full_text.split("Detailed Answer:")[-1].strip()
    else:
        response = full_text[len(prompt):].strip()

    return response


In [7]:
# ──────────────────────────────────────────
# CELL 8: Full End-to-End Pipeline
# ──────────────────────────────────────────
def speech_to_reasoning_pipeline(audio_path):
    """
    FULL PIPELINE:
    Audio file → Whisper STT → LLM Reasoning → Text Answer
    Parameters:
        audio_path: path to audio file
    Returns:
        transcription: what was said
        response: LLM's answer
    """
    print("\n" + "="*60)
    print("  SPEECH-TO-REASONING PIPELINE")
    print("="*60)

    # STEP 1: Transcribe audio to text
    print("\n Step 1: Transcribing audio...")
    transcription = transcribe_audio(audio_path)

    if transcription.startswith("ERROR"): # Check if transcription failed
        return transcription, "Transcription failed, unable to generate AI response."

    # STEP 2: Send to LLM for reasoning
    print("\n Step 2: Generating LLM response...")
    # Ensure model and tokenizer are available, re-load if necessary due to kernel reset
    if 'model' not in globals() or 'tokenizer' not in globals():
        print("WARNING: LLM model or tokenizer not found. Attempting to re-load...")
        from unsloth import FastLanguageModel
        global model, tokenizer # Declare as global to update global scope
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
            max_seq_length = 2048,
            dtype = None,
            load_in_4bit = True,
        )
        FastLanguageModel.for_inference(model)
        print("LLM model and tokenizer re-loaded.")

    response = generate_llm_response(transcription)
    print("\n" + "="*60)
    print(" PIPELINE COMPLETE!")
    print("="*60)
    print(f"\n You said: {transcription}")
    print(f"\n AI Answer:\n{response[:500]}")
    return transcription, response
# Run the full pipeline on our sample audio
transcription, answer = speech_to_reasoning_pipeline("/content/sample_question.mp3")


  SPEECH-TO-REASONING PIPELINE

 Step 1: Transcribing audio...
 Transcribing: /content/sample_question.mp3


Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Transcription done!
Transcribed: 'What are the main symptoms of type 2 diabetes and how is it treated?'

 Step 2: Generating LLM response...

 PIPELINE COMPLETE!

 You said: What are the main symptoms of type 2 diabetes and how is it treated?

 AI Answer:
Type 2 diabetes, also known as non-insulin-dependent diabetes or adult-onset diabetes, is a chronic metabolic disorder characterized by high blood sugar levels. The primary symptoms of type 2 diabetes include:

1. Increased thirst and urination: As the body tries to eliminate excess glucose through urine, individuals with type 2 diabetes may experience increased thirst and frequent trips to the bathroom.
2. Fatigue: Elevated blood sugar levels can cause fatigue, weakness, and lethargy due to the


In [8]:
# ──────────────────────────────────────────
# CELL 9: Test with Multiple Audio Files
# ──────────────────────────────────────────

# Create more test audio files
test_questions = [
    "What are the warning signs of a stroke?",
    "Can you explain how vaccines work?",
    "What is the difference between a virus and bacteria?",
]

for i, q in enumerate(test_questions):
    audio_file = f"/content/test_audio_{i+1}.mp3"

    # Create audio file
    tts = gTTS(text=q, lang='en', slow=False)
    tts.save(audio_file)

    # Run pipeline
    print(f"\n{'='*60}")
    print(f"TEST {i+1}")
    trans, resp = speech_to_reasoning_pipeline(audio_file)
    print(f"\nFull answer preview: {resp[:200]}...")


TEST 1

  SPEECH-TO-REASONING PIPELINE

 Step 1: Transcribing audio...
 Transcribing: /content/test_audio_1.mp3


Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Transcription done!
Transcribed: 'What are the warning signs of a stroke?'

 Step 2: Generating LLM response...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



 PIPELINE COMPLETE!

 You said: What are the warning signs of a stroke?

 AI Answer:
A stroke is a medical emergency that requires immediate attention. Recognizing the warning signs can help you or someone else receive timely treatment, which may significantly improve outcomes. Here are the common warning signs of a stroke:

1. **Sudden weakness or numbness**: If one side of your body feels weak, numb, or tingling, it could be a sign of a stroke.
2. **Sudden confusion or trouble speaking**: Difficulty finding words, understanding others, or making sentences can indicate a stroke

Full answer preview: A stroke is a medical emergency that requires immediate attention. Recognizing the warning signs can help you or someone else receive timely treatment, which may significantly improve outcomes. Here a...

TEST 2

  SPEECH-TO-REASONING PIPELINE

 Step 1: Transcribing audio...
 Transcribing: /content/test_audio_2.mp3


...


 PIPELINE COMPLETE!

 You said: Can you explain how vaccines work?

 AI Answer:
Vaccines are an incredible tool in the fight against infectious diseases. At their core, they work by introducing a small, harmless piece of a virus or bacteria to our immune system, which then learns to recognize and remember that specific pathogen.

When we receive a vaccine, the tiny amount of virus or bacteria is not enough to cause illness, but it's enough for our immune system to detect and respond to. This response triggers the production of antibodies, which are specialized proteins desi

Full answer preview: Vaccines are an incredible tool in the fight against infectious diseases. At their core, they work by introducing a small, harmless piece of a virus or bacteria to our immune system, which then learns...

TEST 3

  SPEECH-TO-REASONING PIPELINE

 Step 1: Transcribing audio...
 Transcribing: /content/test_audio_3.mp3


Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Transcription done!
Transcribed: 'What is the difference between a virus and bacteria?'

 Step 2: Generating LLM response...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



 PIPELINE COMPLETE!

 You said: What is the difference between a virus and bacteria?

 AI Answer:
A virus and bacteria are two types of microorganisms that can cause infections in humans, but they have distinct characteristics that set them apart.

**Virus:**

1. **Lack of cellular structure**: Viruses do not have cells like bacteria. They consist of genetic material (DNA or RNA) surrounded by a protein coat called a capsid.
2. **Replication mechanism**: Viruses replicate inside the host cell's cytoplasm using the host cell's machinery to produce new viral particles.
3. **No metabolism**: Vi

Full answer preview: A virus and bacteria are two types of microorganisms that can cause infections in humans, but they have distinct characteristics that set them apart.

**Virus:**

1. **Lack of cellular structure**: Vi...
